##Part 3 — Machine Learning: predict no-shows and choose the model honestly

In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, precision_score,
    recall_score, f1_score, roc_auc_score
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score, cross_val_predict
from sklearn.pipeline import Pipeline


Task 3.1 — Features, split and baseline

In [3]:
df = pd.read_csv("appointments_clean.csv")

target_col = "no_show"
feature_cols = [
    "age", "distance_km", "fee_inr", "lead_days", "prior_no_shows",
    "reminder_sent", "booking_channel", "visit_type"
]

X = df[feature_cols].copy()
y = df[target_col].copy()

X["reminder_sent"] = X["reminder_sent"].map({"Yes": 1, "No": 0})

X = pd.get_dummies(X, columns=["booking_channel", "visit_type"], drop_first=False)

X = X.astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

train_size = X_train.shape[0]
test_size = X_test.shape[0]
test_no_shows = int(y_test.sum())

baseline_accuracy = (y_test == 0).mean()

print(f"Training set size: {train_size}")
print(f"Test set size: {test_size}")
print(f"Number of no-shows in test set: {test_no_shows}")
print(f"Baseline accuracy (always predicting 'Attended'): {baseline_accuracy:.4f}")

Training set size: 792
Test set size: 198
Number of no-shows in test set: 49
Baseline accuracy (always predicting 'Attended'): 0.7525


Task 3.2 — Logistic regression

In [4]:

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_s, y_train)

y_pred = lr_model.predict(X_test_s)
y_prob = lr_model.predict_proba(X_test_s)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, y_prob)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")
print("\nConfusion Matrix:")
print(conf_matrix)

print("\nModel Coefficients:")
for feature, coef in zip(X.columns, lr_model.coef_[0]):
    print(f"{feature}: {coef:.4f}")

reminder_idx = list(X.columns).index("reminder_sent")
coef_scaled = lr_model.coef_[0][reminder_idx]
std_dev = scaler.scale_[reminder_idx]

or_scaled = np.exp(coef_scaled)

coef_unscaled = coef_scaled / std_dev
or_unscaled = np.exp(coef_unscaled)

print(f"\n--- Reminder Sent Effects ---")
print(f"Scaled Coefficient: {coef_scaled:.4f} | Odds Ratio (scaled): {or_scaled:.4f}")
print(f"Unscaled Coefficient (unit change 0 to 1): {coef_unscaled:.4f} | Odds Ratio (unscaled): {or_unscaled:.4f}")

Accuracy:  0.7879
Precision: 0.6296
Recall:    0.3469
F1 Score:  0.4474
ROC-AUC:   0.7893

Confusion Matrix:
[[139  10]
 [ 32  17]]

Model Coefficients:
age: -0.2551
distance_km: 0.2651
fee_inr: 0.4523
lead_days: 0.6153
prior_no_shows: 0.6509
reminder_sent: -0.5817
booking_channel_App: -0.0469
booking_channel_Phone: 0.2160
booking_channel_Walk-In: -0.1864
visit_type_Consultation: 0.0157
visit_type_Follow-up: -0.0157

--- Reminder Sent Effects ---
Scaled Coefficient: -0.5817 | Odds Ratio (scaled): 0.5589
Unscaled Coefficient (unit change 0 to 1): -1.1752 | Odds Ratio (unscaled): 0.3087


Task 3.3 — Decision tree and overfitting

In [6]:

dt_no_depth = DecisionTreeClassifier(random_state=42)
dt_no_depth.fit(X_train, y_train)

train_acc_no_depth = accuracy_score(y_train, dt_no_depth.predict(X_train))
test_acc_no_depth = accuracy_score(y_test, dt_no_depth.predict(X_test))

dt_depth3 = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_depth3.fit(X_train, y_train)

train_acc_depth3 = accuracy_score(y_train, dt_depth3.predict(X_train))
test_acc_depth3 = accuracy_score(y_test, dt_depth3.predict(X_test))

print("--- Unconstrained Decision Tree ---")
print(f"Training Accuracy: {train_acc_no_depth:.4f}")
print(f"Test Accuracy:     {test_acc_no_depth:.4f}")

print("\n--- Decision Tree (max_depth=3) ---")
print(f"Training Accuracy: {train_acc_depth3:.4f}")
print(f"Test Accuracy:     {test_acc_depth3:.4f}")

--- Unconstrained Decision Tree ---
Training Accuracy: 1.0000
Test Accuracy:     0.6717

--- Decision Tree (max_depth=3) ---
Training Accuracy: 0.8220
Test Accuracy:     0.7778


Task 3.4 — Random forest and a comparison table

In [9]:

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

rf_acc = accuracy_score(y_test, rf_pred)
rf_rec = recall_score(y_test, rf_pred)
rf_auc = roc_auc_score(y_test, rf_prob)

print(f"Random Forest Accuracy:  {rf_acc:.4f}")
print(f"Random Forest Recall:    {rf_rec:.4f}")
print(f"Random Forest ROC-AUC:   {rf_auc:.4f}")

importances = pd.Series(rf_model.feature_importances_, index=X.columns)
top_3_features = importances.nlargest(3)
print("\nTop 3 Feature Importances:")
print(top_3_features)

Random Forest Accuracy:  0.7778
Random Forest Recall:    0.3469
Random Forest ROC-AUC:   0.7064

Top 3 Feature Importances:
lead_days      0.226710
age            0.216596
distance_km    0.165762
dtype: float64


Task 3.5 — Tune, choose the model and the threshold, then test once

In [11]:
# (a) GridSearchCV for Random Forest
rf_base = RandomForestClassifier(random_state=42)
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5, 8]
}

grid_search = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid,
    cv=5,
    scoring="roc_auc"
)
grid_search.fit(X_train, y_train)

print("--- (a) Random Forest Grid Search ---")
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best CV ROC-AUC: {grid_search.best_score_:.4f}\n")

# (b) Cross-Validated Model Selection
# 1. Logistic Regression Pipeline
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])
lr_cv_auc = cross_val_score(lr_pipe, X_train, y_train, cv=5, scoring="roc_auc").mean()

# 2. Decision Tree (max_depth=3)
dt_cv = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_cv_auc = cross_val_score(dt_cv, X_train, y_train, cv=5, scoring="roc_auc").mean()

# 3. Default Random Forest
rf_def = RandomForestClassifier(n_estimators=100, random_state=42)
rf_def_auc = cross_val_score(rf_def, X_train, y_train, cv=5, scoring="roc_auc").mean()

# 4. Tuned Random Forest best score from grid search
tuned_rf_auc = grid_search.best_score_

print("--- (b) Model Comparison (5-Fold CV ROC-AUC) ---")
print(f"Logistic Regression Pipeline: {lr_cv_auc:.4f}")
print(f"Decision Tree (max_depth=3):  {dt_cv_auc:.4f}")
print(f"Random Forest (Default):      {rf_def_auc:.4f}")
print(f"Random Forest (Tuned):        {tuned_rf_auc:.4f}")
print(f"Winner: Logistic Regression Pipeline (Highest CV ROC-AUC)\n")

# (c) Out-of-Fold Threshold Optimization
oof_probs = cross_val_predict(lr_pipe, X_train, y_train, cv=5, method="predict_proba")[:, 1]

print("--- (c) Threshold Evaluation on Out-of-Fold Predictions ---")
for threshold in [0.5, 0.35]:
    oof_preds = (oof_probs >= threshold).astype(int)
    p = precision_score(y_train, oof_preds, zero_division=0)
    r = recall_score(y_train, oof_preds, zero_division=0)
    f = f1_score(y_train, oof_preds, zero_division=0)
    flagged = oof_preds.sum()
    print(f"Threshold {threshold}: Precision={p:.4f}, Recall={r:.4f}, F1={f:.4f}, Flagged Count={flagged}")

chosen_threshold = 0.35
print(f"\nChosen Threshold: {chosen_threshold} (Higher F1 score, balancing precision and recall for operational use)\n")

# (d) Final Evaluation on Test Set
lr_pipe.fit(X_train, y_train)
test_probs = lr_pipe.predict_proba(X_test)[:, 1]
test_preds = (test_probs >= chosen_threshold).astype(int)

test_precision = precision_score(y_test, test_preds, zero_division=0)
test_recall = recall_score(y_test, test_preds, zero_division=0)
test_flagged = int(test_preds.sum())
test_roc_auc = roc_auc_score(y_test, test_probs)

print("--- (d) Final Test Set Performance at Threshold 0.35 ---")
print(f"Precision:    {test_precision:.4f}")
print(f"Recall:       {test_recall:.4f}")
print(f"Number Flagged: {test_flagged}")
print(f"ROC-AUC:      {test_roc_auc:.4f}\n")

# (e) Save test_risk_scores.csv
test_appointment_ids = df.loc[X_test.index, "appointment_id"]
test_actuals = y_test.values

results_df = pd.DataFrame({
    "appointment_id": test_appointment_ids,
    "no_show": test_actuals,
    "risk_prob": test_probs,
    "flag": test_preds
})

results_df.to_csv("test_risk_scores.csv", index=False)
print("Saved test_risk_scores.csv successfully with required columns.")

--- (a) Random Forest Grid Search ---
Best Parameters: {'max_depth': 5, 'n_estimators': 200}
Best CV ROC-AUC: 0.7706

--- (b) Model Comparison (5-Fold CV ROC-AUC) ---
Logistic Regression Pipeline: 0.7744
Decision Tree (max_depth=3):  0.6788
Random Forest (Default):      0.7404
Random Forest (Tuned):        0.7706
Winner: Logistic Regression Pipeline (Highest CV ROC-AUC)

--- (c) Threshold Evaluation on Out-of-Fold Predictions ---
Threshold 0.5: Precision=0.5811, Recall=0.2544, F1=0.3539, Flagged Count=74
Threshold 0.35: Precision=0.5132, Recall=0.4615, F1=0.4860, Flagged Count=152

Chosen Threshold: 0.35 (Higher F1 score, balancing precision and recall for operational use)

--- (d) Final Test Set Performance at Threshold 0.35 ---
Precision:    0.5714
Recall:       0.5714
Number Flagged: 49
ROC-AUC:      0.7893

Saved test_risk_scores.csv successfully with required columns.


In [ ]:
# Why Recall Matters Here: Recall is crucial because a missed no-show warning results in an empty clinic slot and
#  lost revenue, whereas an extra phone call only costs about a minute of staff time.

# Real-World Threshold Selection: A clinic would choose the operating threshold based on staff capacity—specifically,
#  how many reminder phone calls the front desk can realistically make in an evening balanced against the financial and
#  operational cost of an empty appointment slot.